# Step 2 (revised) — Type ground truth with expanded schema

Two changes since the previous version:

1. **Four new fields** to capture information that was being lost:
   - `residence_qualifier` — "over", "nr", "cor", "es", "ws", "bt", etc.
   - `business_name` — for business/firm entries with no personal name
   - `entry_type` — person / business / institution / cross_reference
   - `notes` — free text for anything else (widow status, cross-references, unusual remarks)

2. **Explicit guidance to type the printed form**, not an interpretation.
   The pipeline expands abbreviations later using lookup tables. Ground truth should record what's on the page so evaluation compares like to like.

## The rule for every field

**Type what's printed.** If the page says `(c)`, type `(c)` — not "colored" or "Colored". If the page says `servt`, type `servt` — not "Servant". If the page says `wid Wm. L.`, put that in **notes**, not "widow of William L.".

## Blank = "not on the page"

Empty fields mean the source didn't print that information. Don't type "N/A" or "Not specified" — just leave the field empty.

## Skip toggle for non-entries

If a box is actually an ad slice or a page header, tick "Not a real entry" instead of filling in blank fields.

## 0. Setup — same folders as before

In [5]:
from pathlib import Path
import json
import cv2

STEP1_DIR = Path("output_box_fixing")
CORRECTED_BOXES_PATH = STEP1_DIR / "corrected_boxes.json"

STEP2_DIR = Path("output_ground_truth")
STEP2_DIR.mkdir(exist_ok=True)
(STEP2_DIR / "crops").mkdir(exist_ok=True)

assert CORRECTED_BOXES_PATH.exists(), f"Not found: {CORRECTED_BOXES_PATH}"
print("Setup OK")

Setup OK


## 1. Load boxes and (re)generate crops

In [6]:
data = json.loads(CORRECTED_BOXES_PATH.read_text())
boxes = data["boxes"]

page_image_path = Path(data["page_image"].replace("file:///", "").replace("file://", ""))
if not page_image_path.exists():
    fallback = Path(r"C:\Users\ABHIRAMI.K\Documents\RICE\Summer 2026\Fondren Internship\Data\image\1900-1901 (page 200).png")
    if fallback.exists():
        page_image_path = fallback

page_img = cv2.imread(str(page_image_path))
assert page_img is not None, f"Could not load page image: {page_image_path}"

for b in boxes:
    x1, y1, x2, y2 = int(b["x1"]), int(b["y1"]), int(b["x2"]), int(b["y2"])
    crop = page_img[y1:y2, x1:x2]
    bid = b["id"]
    crop_path = STEP2_DIR / "crops" / f"box_{bid:03d}.png"
    cv2.imwrite(str(crop_path), crop)

print(f"Loaded {len(boxes)} boxes and generated {len(boxes)} crops.")

Loaded 79 boxes and generated 79 crops.


## 2. Build the expanded data-entry tool

The tool now includes the four new fields and a reference panel at the top with sample entries — so the raw-form convention is always visible while typing.

In [7]:
def build_ground_truth_html_v2(boxes, page_image_path, output_path):
    """Two-panel data-entry tool with expanded 13-field schema."""
    page_uri = Path(page_image_path).resolve().as_uri()

    entries_data = []
    for b in boxes:
        crop_path = STEP2_DIR / "crops" / f"box_{b['id']:03d}.png"
        entries_data.append({
            "box_id":    b["id"],
            "crop_uri":  crop_path.resolve().as_uri(),
            "x1": int(b["x1"]), "y1": int(b["y1"]),
            "x2": int(b["x2"]), "y2": int(b["y2"]),
        })

    html = r"""<!DOCTYPE html>
<html><head><meta charset='utf-8'>
<title>Ground truth data entry v2</title>
<style>
  * { box-sizing: border-box; }
  body { font-family: Calibri, Arial, sans-serif; margin: 0; background: #f4f4f4; font-size: 14px; }
  .layout { display: flex; height: 100vh; overflow: hidden; }

  .left-panel { flex: 0 0 40%; background: #eaeaea; padding: 10px; display: flex; flex-direction: column; }
  .left-controls { background: white; padding: 8px 12px; margin-bottom: 8px;
                   border-radius: 6px; font-size: 13px; display: flex; gap: 10px; align-items: center; }
  .left-controls input[type=range] { flex: 1; }
  .page-viewport { flex: 1; background: white; padding: 8px; border-radius: 6px; overflow: auto; }
  .page-wrap { position: relative; display: inline-block; }
  .page-wrap img { display: block; transform-origin: top left; }
  .page-wrap .highlight { position: absolute; border: 3px solid #EF4444;
                          pointer-events: none; transform-origin: top left; }

  .right-panel { flex: 1; padding: 14px 18px; overflow: auto; background: white; }

  .rule-box { background: #FEF3C7; border-left: 4px solid #F59E0B;
              padding: 10px 14px; margin-bottom: 14px; border-radius: 4px; font-size: 13px; }
  .rule-box b { color: #92400E; }
  .rule-box code { background: white; padding: 1px 5px; border-radius: 3px;
                   font-family: 'Courier New', monospace; font-size: 12px; }
  details.reference { background: #E8F4F1; border-radius: 4px; padding: 8px 14px;
                      margin-bottom: 14px; font-size: 13px; }
  details.reference summary { cursor: pointer; color: #1D6E5E; font-weight: bold; }
  details.reference table { margin-top: 8px; border-collapse: collapse; font-size: 12px; }
  details.reference th, details.reference td { padding: 4px 8px;
                                                border: 1px solid #D1D5DB; text-align: left; }
  details.reference th { background: white; color: #374151; }

  h1 { color: #1D6E5E; font-size: 18px; margin: 0 0 4px 0; }
  .header-bar { display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px; }
  .progress-info { font-size: 13px; color: #6B7280; }

  button { background: #1D6E5E; color: white; border: none; padding: 8px 14px;
           font-size: 13px; border-radius: 5px; cursor: pointer; font-family: Calibri; margin-left: 6px; }
  button.small { padding: 3px 8px; font-size: 11px; }
  button.secondary { background: #6B7280; }
  button.danger { background: #DC2626; }
  button:hover { opacity: 0.9; }

  .card { background: #fafafa; border: 1px solid #e5e7eb; border-radius: 6px;
          padding: 10px 14px; margin-bottom: 12px; }
  .card.skipped { background: #f3f4f6; opacity: 0.6; }
  .card.done { border-left: 4px solid #10B981; }
  .card-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; }
  .box-id b { color: #1D6E5E; font-size: 14px; }
  .crop { display: block; max-width: 100%; max-height: 90px;
          border: 1px solid #d1d5db; margin-bottom: 8px; }

  .entry { margin-bottom: 8px; padding: 6px 0; }
  .entry:not(:first-child) { border-top: 1px dashed #d1d5db; padding-top: 8px; margin-top: 8px; }
  .entry-label { font-size: 11px; color: #9CA3AF; margin-bottom: 4px; }
  table.fields { width: 100%; border-collapse: collapse; }
  table.fields td { padding: 2px 4px; vertical-align: middle; }
  table.fields td.label { width: 130px; color: #374151; font-size: 12px; }
  table.fields td.label .hint { font-size: 10px; color: #9CA3AF; display: block; }
  table.fields input, table.fields select { width: 100%; border: 1px solid #d1d5db;
                                             padding: 3px 5px; font-family: Calibri; font-size: 13px; }

  .footer { position: sticky; bottom: -14px; background: white; border-top: 2px solid #1D6E5E;
            padding: 12px; margin: 20px -18px -14px -18px; text-align: center; }
</style></head><body>

<div class='layout'>

  <div class='left-panel'>
    <div class='left-controls'>
      <span>Zoom:</span>
      <input type='range' min='30' max='150' value='60' id='zoom' oninput='setZoom(this.value)'>
      <span id='zoom-val'>60%</span>
      <button class='small secondary' onclick='scrollToActive()'>Center on box</button>
    </div>
    <div class='page-viewport' id='viewport'>
      <div class='page-wrap' id='page-wrap'>
        <img id='page-image' src='__PAGE_URI__' alt='page'>
        <div class='highlight' id='highlight' style='display:none'></div>
      </div>
    </div>
  </div>

  <div class='right-panel'>
    <div class='header-bar'>
      <div>
        <h1>Ground truth data entry — page 200</h1>
        <div class='progress-info' id='progress'></div>
      </div>
      <div>
        <button class='secondary' onclick='saveProgress()'>Save draft</button>
        <button onclick='finalize()'>Save final</button>
      </div>
    </div>

    <div class='rule-box'>
      <b>Type what's printed on the page — not an interpretation.</b><br>
      Race: type <code>(c)</code> not "Colored" &nbsp; · &nbsp;
      Occupation: type <code>servt</code> not "Servant" &nbsp; · &nbsp;
      Employer: type <code>Wm. Reichardt</code> not "William Reichardt" <br>
      Blank field means "not on the page" — do not type "N/A" or "None".
    </div>

    <details class='reference'>
      <summary>Reference: 3 example entries typed correctly (click to expand)</summary>

      <p><b>Hatcher Charles</b> — "teamster Hipp &amp; Key, rms over 1219 Hamilton."</p>
      <table>
        <tr><th>last_name</th><td>Hatcher</td>
            <th>occupation</th><td>teamster</td></tr>
        <tr><th>first_name</th><td>Charles</td>
            <th>employer</th><td>Hipp &amp; Key</td></tr>
        <tr><th>race</th><td><i>(blank)</i></td>
            <th>rooms</th><td>rms over 1219 Hamilton</td></tr>
        <tr><th>residence_qualifier</th><td>over</td>
            <th>entry_type</th><td>person</td></tr>
      </table>

      <p><b>Hatcher Sallie (c)</b> — "servt Wm. Reichardt, r. same."</p>
      <table>
        <tr><th>last_name</th><td>Hatcher</td>
            <th>occupation</th><td>servt</td></tr>
        <tr><th>first_name</th><td>Sallie</td>
            <th>employer</th><td>Wm. Reichardt</td></tr>
        <tr><th>race</th><td>(c)</td>
            <th>residence</th><td>r. same</td></tr>
        <tr><th>entry_type</th><td>person</td>
            <th>notes</th><td><i>(blank)</i></td></tr>
      </table>

      <p><b>Hatfield &amp; Hopkins</b> — "(Edward T. Hatfield, Earl P. Hopkins), attorneys, office 1009½ Congress ave."</p>
      <table>
        <tr><th>last_name</th><td><i>(blank)</i></td>
            <th>occupation</th><td>attorneys</td></tr>
        <tr><th>first_name</th><td><i>(blank)</i></td>
            <th>employer</th><td><i>(blank)</i></td></tr>
        <tr><th>business_name</th><td>Hatfield &amp; Hopkins</td>
            <th>workplace_address</th><td>office 1009½ Congress ave</td></tr>
        <tr><th>entry_type</th><td>business</td>
            <th>notes</th><td>Edward T. Hatfield, Earl P. Hopkins</td></tr>
      </table>

      <p><b>Hathaway Edward C.</b> — "barber, shop 1102 McKee, r. 904 McKee."</p>
      <table>
        <tr><th>last_name</th><td>Hathaway</td>
            <th>occupation</th><td>barber</td></tr>
        <tr><th>first_name</th><td>Edward C.</td>
            <th>employer</th><td><i>(blank)</i></td></tr>
        <tr><th>workplace_address</th><td>shop 1102 McKee</td>
            <th>residence</th><td>r. 904 McKee</td></tr>
        <tr><th>entry_type</th><td>person</td>
            <th>notes</th><td><i>(blank)</i></td></tr>
      </table>
    </details>

    <div id='cards'></div>

    <div class='footer'>
      <button class='secondary' onclick='saveProgress()'>Save draft</button>
      <button onclick='finalize()'>Save ground truth (final)</button>
    </div>
  </div>

</div>

<script>
const BOXES = __BOXES_JSON__;

const FIELDS = [
  {key: 'last_name',           label: 'Last name',           hint: 'surname of person'},
  {key: 'first_name',          label: 'First name',          hint: 'may include middle initial'},
  {key: 'business_name',       label: 'Business name',       hint: 'for firm/business entries only'},
  {key: 'race',                label: 'Race',                hint: 'raw marker e.g. (c)'},
  {key: 'occupation',          label: 'Occupation',          hint: 'raw e.g. lab, clk, servt'},
  {key: 'employer',            label: 'Employer',            hint: 'raw e.g. S. P. shops'},
  {key: 'workplace_address',   label: 'Workplace address',   hint: 'raw e.g. shop 1102 McKee, office 1009\u00bd Congress ave'},
  {key: 'residence',           label: 'Residence',           hint: 'raw e.g. r. 1902 Jackson'},
  {key: 'boarding',            label: 'Boarding',            hint: 'raw e.g. bds 803 Main'},
  {key: 'rooms',               label: 'Rooms',               hint: 'raw e.g. rms over 1219 Hamilton'},
  {key: 'residence_qualifier', label: 'Res. qualifier',      hint: 'over, nr, cor, es, ws, bt, etc.'},
  {key: 'ownership',           label: 'Ownership',           hint: 'home, householder, or blank'},
  {key: 'entry_type',          label: 'Entry type',          hint: 'person / business / institution / cross_reference'},
  {key: 'notes',               label: 'Notes',               hint: 'widow status, cross-refs, remarks'},
];

let state = {};
BOXES.forEach(b => {
  state[b.box_id] = { skipped: false, entries: [newEmptyEntry()] };
});

let activeBoxId = null;
let zoom = 0.6;

function newEmptyEntry() {
  const e = {};
  FIELDS.forEach(f => { e[f.key] = f.key === 'entry_type' ? 'person' : ''; });
  return e;
}

function escAttr(s) { return String(s || '').replace(/&/g,'&amp;').replace(/'/g,'&#39;').replace(/"/g,'&quot;'); }
function escHtml(s) { return String(s || '').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;'); }

function setZoom(v) {
  zoom = v / 100;
  document.getElementById('zoom-val').textContent = v + '%';
  const img = document.getElementById('page-image');
  img.style.transform = 'scale(' + zoom + ')';
  document.getElementById('page-wrap').style.width  = (img.naturalWidth  * zoom) + 'px';
  document.getElementById('page-wrap').style.height = (img.naturalHeight * zoom) + 'px';
  updateHighlight();
}

function updateHighlight() {
  const el = document.getElementById('highlight');
  if (activeBoxId === null) { el.style.display = 'none'; return; }
  const b = BOXES.find(x => x.box_id === activeBoxId);
  if (!b) { el.style.display = 'none'; return; }
  el.style.display = 'block';
  el.style.left   = (b.x1 * zoom) + 'px';
  el.style.top    = (b.y1 * zoom) + 'px';
  el.style.width  = ((b.x2 - b.x1) * zoom) + 'px';
  el.style.height = ((b.y2 - b.y1) * zoom) + 'px';
}

function scrollToActive() {
  if (activeBoxId === null) return;
  const b = BOXES.find(x => x.box_id === activeBoxId);
  if (!b) return;
  const vp = document.getElementById('viewport');
  const centerY = b.y1 * zoom - vp.clientHeight / 2 + ((b.y2 - b.y1) * zoom) / 2;
  const centerX = b.x1 * zoom - vp.clientWidth  / 2 + ((b.x2 - b.x1) * zoom) / 2;
  vp.scrollTo({ top: Math.max(0, centerY), left: Math.max(0, centerX), behavior: 'smooth' });
}

function setActive(boxId) {
  activeBoxId = boxId;
  updateHighlight();
  scrollToActive();
}

function render() {
  const container = document.getElementById('cards');
  container.innerHTML = '';
  BOXES.forEach(b => {
    const s = state[b.box_id];
    const card = document.createElement('div');
    card.className = 'card' + (s.skipped ? ' skipped' : (isEntryFilled(s.entries[0]) ? ' done' : ''));
    card.onclick = () => setActive(b.box_id);

    let entriesHtml = '';
    s.entries.forEach((entry, idx) => {
      let rows = '';
      FIELDS.forEach(f => {
        let input;
        if (f.key === 'entry_type') {
          input = `<select onchange='updateField(${b.box_id}, ${idx}, "${f.key}", this.value)'>
            <option value='person' ${entry[f.key]==='person'?'selected':''}>person</option>
            <option value='business' ${entry[f.key]==='business'?'selected':''}>business</option>
            <option value='institution' ${entry[f.key]==='institution'?'selected':''}>institution</option>
            <option value='cross_reference' ${entry[f.key]==='cross_reference'?'selected':''}>cross_reference</option>
          </select>`;
        } else if (f.key === 'ownership') {
          input = `<select onchange='updateField(${b.box_id}, ${idx}, "${f.key}", this.value)'>
            <option value='' ${!entry[f.key]?'selected':''}>(blank)</option>
            <option value='home' ${entry[f.key]==='home'?'selected':''}>home</option>
            <option value='householder' ${entry[f.key]==='householder'?'selected':''}>householder</option>
          </select>`;
        } else {
          input = `<input type='text' value='${escAttr(entry[f.key])}'
                    oninput='updateField(${b.box_id}, ${idx}, "${f.key}", this.value)'>`;
        }
        rows += `<tr>
          <td class='label'>${f.label}<span class='hint'>${escHtml(f.hint)}</span></td>
          <td>${input}</td>
        </tr>`;
      });
      const entryLabel = s.entries.length > 1
        ? `<div class='entry-label'>Entry ${idx + 1} of ${s.entries.length}</div>` : '';
      const removeBtn = s.entries.length > 1
        ? `<div style='margin-top:6px;'><button class='small danger' onclick='event.stopPropagation(); removeEntry(${b.box_id}, ${idx})'>Remove entry ${idx + 1}</button></div>` : '';
      entriesHtml += `<div class='entry'>${entryLabel}<table class='fields'>${rows}</table>${removeBtn}</div>`;
    });

    card.innerHTML = `
      <div class='card-header'>
        <div class='box-id'>Box <b>#${b.box_id}</b></div>
        <div>
          <button class='small secondary' onclick='event.stopPropagation(); addEntry(${b.box_id})'>+ Add entry (box has multiple)</button>
          <label style='margin-left:8px;font-size:12px;'>
            <input type='checkbox' ${s.skipped ? 'checked' : ''}
                   onchange='toggleSkip(${b.box_id}, this.checked)'>
            Not a real entry
          </label>
        </div>
      </div>
      <img src='${b.crop_uri}' class='crop' alt='crop'>
      ${entriesHtml}
    `;
    container.appendChild(card);
  });
  updateProgress();
}

function isEntryFilled(entry) {
  return !!(entry.last_name || entry.first_name || entry.business_name || entry.occupation);
}

function updateField(boxId, entryIdx, field, value) {
  state[boxId].entries[entryIdx][field] = value;
  updateProgress();
  updateCardStatus(boxId);
}

function updateCardStatus(boxId) {
  const cards = document.getElementById('cards').children;
  BOXES.forEach((b, i) => {
    if (b.box_id !== boxId) return;
    const s = state[boxId];
    const card = cards[i];
    card.classList.remove('skipped', 'done');
    if (s.skipped) card.classList.add('skipped');
    else if (isEntryFilled(s.entries[0])) card.classList.add('done');
  });
}

function addEntry(boxId) { state[boxId].entries.push(newEmptyEntry()); render(); }
function removeEntry(boxId, idx) {
  if (state[boxId].entries.length <= 1) return;
  state[boxId].entries.splice(idx, 1);
  render();
}
function toggleSkip(boxId, checked) {
  state[boxId].skipped = checked;
  updateCardStatus(boxId);
  updateProgress();
}

function updateProgress() {
  let filled = 0, skipped = 0;
  BOXES.forEach(b => {
    const s = state[b.box_id];
    if (s.skipped) skipped++;
    else if (isEntryFilled(s.entries[0])) filled++;
  });
  const pending = BOXES.length - filled - skipped;
  document.getElementById('progress').textContent =
    `${filled} filled  ·  ${skipped} skipped  ·  ${pending} pending  ·  ${BOXES.length} total`;
}

function saveProgress() {
  const payload = { boxes: BOXES, state: state, schema_version: 'v2' };
  downloadJson(payload, 'ground_truth_draft.json');
  alert('Draft saved. Reopen the HTML and paste back to resume, or move on.');
}

function finalize() {
  const filled = BOXES.filter(b => !state[b.box_id].skipped &&
                                  isEntryFilled(state[b.box_id].entries[0])).length;
  const skipped = BOXES.filter(b => state[b.box_id].skipped).length;
  const total = BOXES.length;
  if (filled + skipped < total) {
    if (!confirm(`Only ${filled + skipped} of ${total} boxes have been reviewed. Save anyway?`)) return;
  }
  const ground_truth = BOXES.map(b => {
    const s = state[b.box_id];
    return {
      box_id: b.box_id,
      box_page: [b.x1, b.y1, b.x2, b.y2],
      skipped: s.skipped,
      entries: s.skipped ? [] : s.entries.filter(e => isEntryFilled(e)),
    };
  });
  const payload = { schema_version: 'v2', page: '1900_p200', boxes: ground_truth };
  downloadJson(payload, 'ground_truth.json');
  alert('Ground truth saved. Move ground_truth.json into output_ground_truth/ and run the next cell.');
}

function downloadJson(obj, filename) {
  const blob = new Blob([JSON.stringify(obj, null, 2)], {type: 'application/json'});
  const url = URL.createObjectURL(blob);
  const a = document.createElement('a');
  a.href = url; a.download = filename; a.click();
  URL.revokeObjectURL(url);
}

document.getElementById('page-image').addEventListener('load', () => { setZoom(60); });
render();
</script>
</body></html>"""

    html = (html
            .replace("__PAGE_URI__", page_uri)
            .replace("__BOXES_JSON__", json.dumps(entries_data)))
    output_path.write_text(html, encoding="utf-8")
    return output_path


tool_html = build_ground_truth_html_v2(
    boxes, page_image_path, STEP2_DIR / "data_entry_v2.html"
)
print(f"Data entry tool written:\n  {tool_html.resolve()}")
print()
print("Open in Chrome/Firefox. The yellow rule box at the top explains the raw-form convention.")
print("Click the expandable 'Reference' section to see three worked examples.")

Data entry tool written:
  C:\Users\ABHIRAMI.K\Downloads\output_ground_truth\data_entry_v2.html

Open in Chrome/Firefox. The yellow rule box at the top explains the raw-form convention.
Click the expandable 'Reference' section to see three worked examples.


## 3. After saving — sanity check

After clicking "Save ground truth (final)" and moving `ground_truth.json` into `output_ground_truth/`, this cell loads it back and shows a schema-aware summary.

In [8]:
gt_path = STEP2_DIR / "ground_truth.json"

if not gt_path.exists():
    print(f"Not found yet: {gt_path}")
else:
    gt_data = json.loads(gt_path.read_text())
    schema_ver = gt_data.get("schema_version", "unknown")
    boxes_gt = gt_data["boxes"]

    total_boxes = len(boxes_gt)
    skipped_boxes = sum(1 for g in boxes_gt if g["skipped"])
    real_entries = [e for g in boxes_gt for e in g["entries"]]
    multi_entry_boxes = [g for g in boxes_gt if len(g["entries"]) > 1]

    print(f"Schema version:            {schema_ver}")
    print(f"Total boxes:               {total_boxes}")
    print(f"Boxes marked non-entry:    {skipped_boxes}")
    print(f"Real entries captured:     {len(real_entries)}")
    print(f"Boxes containing >1 entry: {len(multi_entry_boxes)}")

    # Count by entry_type
    from collections import Counter
    type_counts = Counter(e.get("entry_type", "person") for e in real_entries)
    print(f"\\nEntry types:")
    for t, n in type_counts.most_common():
        print(f"  {t}: {n}")

    # Show the first three
    print(f"\\nFirst three real entries:\\n")
    for i, entry in enumerate(real_entries[:3]):
        print(f"  Entry {i+1}:")
        for k, v in entry.items():
            if v:
                print(f"    {k:22s}: {v}")
        print()

    print("Ground truth ready. Fields captured in raw printed form —")
    print("suitable for direct comparison against Gemini output in step 3.")

Schema version:            v2
Total boxes:               79
Boxes marked non-entry:    0
Real entries captured:     81
Boxes containing >1 entry: 2
\nEntry types:
  person: 78
  business: 1
  institution: 1
  cross_reference: 1
\nFirst three real entries:\n
  Entry 1:
    last_name             : Hatcher
    first_name            : Charles
    occupation            : teamster
    employer              : Hipp & Key
    rooms                 : rms over 1219 Hamilton
    residence_qualifier   : over
    entry_type            : person

  Entry 2:
    last_name             : Hatcher
    first_name            : Gracie
    occupation            : milliner
    employer              : Alkemeyer Co.
    residence             : r. 617 Girard
    entry_type            : person
    notes                 : Miss

  Entry 3:
    last_name             : Hatcher
    first_name            : Sallie
    race                  : (c)
    occupation            : servt 
    employer              : Wm. Reichardt
